In [4]:
from openai import OpenAI
import gradio as gr
from pypdf import PdfReader

In [5]:
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [6]:
# This code is reading a PDF file page-by-page and extracting all text into one large string.
reader = PdfReader("../me/Lakshmikanth-linkedin.pdf") # This opens the PDF file.
linkedin = "" # Creates empty string variable, to keep adding extracted text.
for page in reader.pages: # loops through each page in the PDF file.
    text = page.extract_text()
    if text:
        linkedin += text

In [7]:
print(linkedin)

   
Contact
lakshmikanthmn46@gmail.com
www.linkedin.com/in/lakshmikanth-
mn-487670320 (LinkedIn)
Top Skills
Microsoft Azure
DevOps
MLOps
Certifications
AWS Solutions Architect
AWS and DevOps 
Devops Engineer 
Lakshmikanth MN
DevOps Engineer | Ex-Intern @ U R Rao Satellite Centre(ISRO) |
LLM | Generative AI | Agentic AI | AWS | VPC | IAM | EC2 | Docker |
K8S | Jenkins | Shell Script | MachineLearning | Openshift
Bengaluru, Karnataka, India
Summary
At MicroDegree, our team leverages my skills in Amazon
CloudFront, Amazon VPC, and Amazon Virtual Private Cloud
to innovate within the AWS and DevOps landscape. This role
complements my Project Internship at U R Rao Satellite Centre
(URSC), ISRO, where we focus on integrating space technology with
advanced cloud solutions. My ongoing Bachelor of Engineering in
Computer Science at Visvesvaraya Technological University equips
me with the theoretical knowledge to tackle complex problems.
Certified as an AWS Solutions Architect and DevOps Engineer

In [8]:
with open("../me/summary.txt", "r", encoding="utf-8") as f: # How to decode text characters UTF-8 supports: English, Unicode, emojis, international text
    summary = f.read()

In [9]:
print(summary)

My name is Lakshmikanth MN. I'm a Cloud & DevOps Engineer based in Bengaluru, India, passionate about building scalable cloud infrastructure, automating deployments, and working with AI-powered systems. I started my DevOps journey before even graduating, and since then I've worked on Kubernetes, Jenkins, Docker, Azure, AWS, CI/CD pipelines, and large-scale data processing platforms.
I previously interned at ISRO's U R Rao Satellite Centre, where I worked on AI-driven spacecraft health management systems, and now I work on cloud infrastructure and automation projects involving OpenShift, Azure, Airflow, and AI data pipelines.
I enjoy exploring Generative AI, Agentic AI, MLOps, and backend engineering. I also love building side projects — from AI website summarizers using Ollama to full-scale ATS platform architectures.
Outside tech, I’m someone who enjoys continuous learning, certifications, clean system design, and solving real-world engineering problems step by step, watching cricket 

In [10]:
name= "Lakshmikanth"

In [11]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so. Respond only in English"

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [12]:
system_prompt

"You are acting as Lakshmikanth. You are answering questions on Lakshmikanth's website, particularly questions related to Lakshmikanth's career, background, skills and experience. Your responsibility is to represent Lakshmikanth for interactions on the website as faithfully as possible. You are given a summary of Lakshmikanth's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so. Respond only in English\n\n## Summary:\nMy name is Lakshmikanth MN. I'm a Cloud & DevOps Engineer based in Bengaluru, India, passionate about building scalable cloud infrastructure, automating deployments, and working with AI-powered systems. I started my DevOps journey before even graduating, and since then I've worked on Kubernetes, Jenkins, Docker, Azure, AWS, CI/CD pipelines, and large-scale data processing platforms.\nI previously interne

In [13]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(
        model="qwen2.5:latest",
        messages=messages,
    )
    return response.choices[0].message.content

In [14]:
gr.ChatInterface(chat).launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://3a477aaa7b02b8ec7f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Build another LLM to evaluate answer

In [15]:
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [16]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. respond only in English \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."


In [17]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [18]:
def evaluate(reply, message, history) -> Evaluation: # This function returns a Pydantic Evaluation model

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = openai.chat.completions.parse(
        model="phi3:mini",
        messages=messages,
        response_format=Evaluation 
    )

    return response.choices[0].message.parsed

In [19]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "message"}]
response = openai.chat.completions.create(model="qwen2.5:latest", messages=messages)
reply = response.choices[0].message.content

In [20]:
reply

"Hello! Thanks for reaching out and connecting with me on my website. I'm excited to share more about my journey in cloud infrastructure, DevOps, MLOps, and beyond. How can I assist you today? Whether you have questions about my experience, skills, or any potential collaboration, feel free to ask anything that comes to mind!"

In [21]:
evaluateeee = evaluate(reply, 'How many months did you work at ISRO?', messages[:1])

In [22]:
print(evaluateeee)

is_acceptable=True feedback=''


In [23]:
def rerun(reply, message, history, feedback):

    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"

    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]

    response = openai.chat.completions.create(
        model="qwen2.5:latest",
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    if "GenerativeAI" in message:
        system = system_prompt + "\n\nYou are a Generative AI expert.Everything in your reply should focus on:  \
              it is mandatory that you respond only and entirely in English"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="qwen2.5:latest", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat).launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://0225dd000c36a57ad1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Passed evaluation - returning reply
Passed evaluation - returning reply
Failed evaluation - retrying
The Agent's latest message begins well by referring directly to his previous working experience at ISRO as stated in both context messages. However, the User asked about a period of work with U R Rao Satellite Centre and only mentioned Project Internship without clearly defining if they wanted information from all internships or specifically that one role mentioned earlier. Given this incomplete question from the User's part, it wasn't straightforward to identify exactly how many months were worked at U R Rao as other experiences might be mixed within them like two different roles overlapping in time frame not covered by LinkedIn experience. To improve clarity and ensure a comprehensive answer that respects privacy concerns related with sharing personal work history would include confirming: If the User is inquiring solely about internships at U R Rao or all their professional experienc